# Model Evaluation # 

This notebook will walk through the process of making N4 predictions using the models trained in this work. 

**INPUT:** An excel file (see `../data/input_format_template.xlsx`) with:
- glass compositions in molar % 
- an associated cooling rate which is one of 'Slow cooled', 'Air quench', 'Water quench', 'Fast quench'. 
- [optional] glass IDs, measured N<sub>4</sub> values to validate predictions against

Ensure that your column names match those in this work. If your compositions have additional components which the models are not trained on, they will be ignored. 

**OUTPUT:** An excel file with the glass compositions and predictions for all models.

In [ ]:
import sys
import os

# Add the parent directory to the Python path to access heteroskedastic DNN and PBNN loading modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import pickle
from ML_Models.heteroscedastic_DNN import HeteroscedasticDNN
from ML_Models.pbnn_save_and_load import load_partial_bnn
from Analytical_Models.Bernstein_DS_Models import calculate_bernstein, calculate_ds
import numpy as np
import neurobayes as nb
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import OrdinalEncoder


# Update to your filename
filename = "../data/boron_coord_final.xlsx"
n4_df = pd.read_excel(filename)
n4_df.head()

## Load trained ML models and required preprocessors 

In [ ]:
with open('./pickles/preprocessor.pkl', 'rb') as file:
    preprocessor = pickle.load(file) 

with open('./pickles/yscale.pkl', 'rb') as file:
    yscale = pickle.load(file) 

with open('./pickles/gpflow.pkl', 'rb') as file:
    gpflow_model = pickle.load(file)

dnn_model = HeteroscedasticDNN.load('.\\pickles\\dnn_model')


In [ ]:
architecture = nb.FlaxMLP(hidden_dims=[32, 32, 32], target_dim=1) # detnn.model
pbnn_model = load_partial_bnn(
    architecture,
    probabilistic_settings={
    "probabilistic_layer_names":['Dense0', f'Dense3']
    },
    folder='./pickles/pbnn')

# Process data for modeling

In [ ]:
cooling_order = [['Slow cooled', 'Air quench', 'Water quench', 'Fast quench']]
enc = OrdinalEncoder(categories=cooling_order)

other = ['Cr2O3', 'HfO2', 'PbO', 'UO3', 'Y2O3','ZrO2','Fe2O3',
         'P2O5', 'TiO2', 'CoO','SrO','Cs2O','Bi2O3','BaO','SO3','La2O3']
oxEl = ['Al2O3','B2O3','CaO','K2O','Li2O','MgO','Na2O','SiO2',
 'ZnO','Rb2O']

input_columns = other + oxEl

In [ ]:
if 'Cooling rate' not in n4_df:
    print('WARNING! Cooling rate needs to be defined in order to continue. Cooling rate can be one of: Slow cooled, Air quench, Water quench, or Fast quench.')

# If a required oxide is not included in your dataset, add the column and assign all rows 0.0 so models can run.
for column in input_columns:
    if column not in n4_df:
        n4_df[column] = 0.0 

# Encode the cooling rate to its numeric value
n4_df['Ordinal cooling'] = enc.fit_transform(n4_df[['Cooling rate']])

In [ ]:
#Format X columns. If you have a measured N4 value you would like to compare the predictions to, you can assign that column to the y variable.
n4_df['other'] = n4_df[other].sum(axis=1)
class_cols = oxEl + ['other', 'Ordinal cooling']
X = n4_df[ class_cols ].fillna(0)

# Optional: measured N4 value to validate predictions.
# y = n4_df['Measured N4']


In [ ]:
# Scale X for ML models
X_scaled = preprocessor.transform(X)
X_scaled.shape

# Make Predictions

## ML Models

In [ ]:
gpr_y_pred_scaled, gpr_y_var_scaled = gpflow_model.predict_y(X_scaled)

In [ ]:
pbnn_y_pred_scaled, pbnn_y_var_scaled = pbnn_model.predict(X_scaled)

In [ ]:
dnn_y_pred, dnn_y_var = dnn_model.predict_with_uncertainty(X)

## Analytical Models

In [ ]:
# Analytical models only require oxides as input features, they will not be scaled. 
X_analytical = n4_df[input_columns]

In [ ]:
bernstein_y_pred = X_analytical.apply(calculate_bernstein, axis = 1)
ds_y_pred = X_analytical.apply(calculate_ds, axis = 1)

In [ ]:
std_var = yscale.scale_

gpr_y_pred = yscale.inverse_transform(gpr_y_pred_scaled).flatten()
gpr_y_var = np.sqrt((gpr_y_var_scaled * std_var).numpy()).flatten()

pbnn_y_pred = yscale.inverse_transform(pbnn_y_pred_scaled).flatten()
pbnn_y_var = (pbnn_y_var_scaled * std_var).flatten()

dnn_y_pred = dnn_y_pred.flatten()
dnn_y_var = dnn_y_var.flatten()

# Save Results #

Save all model results to an excel file

In [ ]:
n4_df['GPR prediction (%)'] = gpr_y_pred
n4_df['GPR variance'] = gpr_y_pred

n4_df['PBNN prediction (%)'] = pbnn_y_pred
n4_df['PBNN variance'] = pbnn_y_pred

n4_df['DNN prediction (%)'] = dnn_y_pred
n4_df['DNN variance'] = dnn_y_pred


n4_df['MB prediction'] = bernstein_y_pred
n4_df['MDX prediction'] = ds_y_pred

n4_df.to_excel('./n4_model_predictions_out.xlsx')

# Visualize Results 

If you have a measured $N_4$ value to compare the predictions to, the following code can be used to help visualize the results.

In [ ]:
color_set = {
    'GPR': sns.color_palette()[3],
    'PBNN': sns.color_palette()[0],
    'DNN': sns.color_palette()[2]
}

marker_set = {
    'GPR': 'o',
    'PBNN': 'd',
    'DNN': '^'
}

In [ ]:
y_test = y
y_preds = [dnn_y_pred, pbnn_y_pred, gpr_y_pred]
y_vars = [dnn_y_var, pbnn_y_var, gpr_y_var]
models = ['DNN', 'PBNN', 'GPR']

plt.figure(figsize=(8,6))
for i, (model, y_pred, var) in enumerate(zip(models, y_preds, y_vars)):
    plt.scatter(y_test, y_pred, alpha=0.6, label=f'{model}, R\u00B2 = {round(r2_score(y_test, y_pred),3)} \n RMSE = {round(root_mean_squared_error(y_test, y_pred),3)}', marker=marker_set[model], s=70, color=color_set[model])
    plt.errorbar(yerr=var, x=y_test, y=y_pred,  fmt='none', ecolor=color_set[model], capsize=3, alpha=0.5)

plt.plot([0, 100], [0, 100], 'r--', linewidth = 2.5)  # Ideal line
plt.xlabel(r"$N_4$ measured (%)")
plt.ylabel(r"$N_4$ predicted (%)")
plt.title(f"ML Model Predictions")
plt.legend()
plt.show()

In [ ]:
y_test = y_test/100

plt.scatter(y_test, ds_y_pred, alpha=0.6, label=f'Modified DS, R\u00B2 = {round(r2_score(y_test, ds_y_pred),3)} \n RMSE = {round(root_mean_squared_error(y_test, ds_y_pred),3)}', color='orange')
plt.scatter(y_test, bernstein_y_pred, alpha=0.6, label=f'Modified Bernstein, R\u00B2 = {round(r2_score(y_test, bernstein_y_pred),3)} \n RMSE = {round(root_mean_squared_error(y_test, bernstein_y_pred),3)}', color='teal')

plt.plot([0, 1], [0, 1], 'r--', linewidth = 2.5)  # Ideal line
plt.title('Analytical Model Predictions')
plt.xlabel(r"$N_4$ measured (%)")
plt.ylabel(r"$N_4$ predicted (%)")
plt.legend()